In [ ]:
# Cell 1: Mount Drive, locate baseline directory, set CWD, check GPU
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive. Please verify path.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA L4


In [ ]:
# Cell 2: Install dependencies via pip (no bash cell) - FIXED for Python 3.12
import sys, subprocess

# 1) Upgrade installer tooling (helps avoid build issues)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"])

# 2) Uninstall potentially conflicting packages (ignore errors if not installed)
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "transformers", "tokenizers", "huggingface-hub",
                 "pandas", "numpy", "tqdm", "pyyaml"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3) Install pinned versions (pandas bumped to a Py3.12-compatible version)
pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",
    "pandas==2.2.2",          # <-- changed from 2.0.3 (not Py3.12 friendly)
    "huggingface-hub==0.21.4",
    "tokenizers==0.15.2",
    "transformers==4.38.1",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import yaml, tqdm, numpy, pandas, transformers, huggingface_hub, tokenizers
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("Installed OK")

huggingface_hub: 0.21.4
tokenizers: 0.15.2
transformers: 4.38.1
numpy: 1.26.4
pandas: 2.2.2
Installed OK


In [ ]:
# Cell 3: Load MDR config and enforce CUDA device
import os, yaml, torch

cfg_path = os.path.join(BASE_DIR, "configs", "MDR.yml")
assert os.path.isfile(cfg_path), f"Missing config: {cfg_path}"

args = yaml.safe_load(open(cfg_path, "r"))

# Force GPU usage in Colab
args["gpus"] = "cuda"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# L4 safety: keep num_workers small to avoid dataloader issues
if "num_workers" not in args:
    args["num_workers"] = 0
args["num_workers"] = int(args["num_workers"])

print("Loaded config:", cfg_path)
print("device =", device)
print("eval_bsz =", args.get("eval_bsz"))
print("max_len/max_q_len/max_q_sp_len =", args["max_len"], args["max_q_len"], args["max_q_sp_len"])

Loaded config: /content/drive/MyDrive/final_project/baseline/configs/MDR.yml
device = cuda
eval_bsz = 32
max_len/max_q_len/max_q_sp_len = 200 64 256


In [ ]:
# Cell 4: Utils and tokenizer loader (same logic)
import random
import numpy as np
import torch
from transformers import AutoConfig, AutoTokenizer

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def load_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    cfg = AutoConfig.from_pretrained(model_name)
    return tok, cfg

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def move_to_gpu(sample, device):
    if len(sample) == 0:
        return {}
    def _move(x):
        if torch.is_tensor(x):
            return x.to(device)
        if isinstance(x, dict):
            return {k: _move(v) for k, v in x.items()}
        return x
    return _move(sample)

seed_everything(args["seed"])

In [ ]:
# Cell 5: Dataset and collate (same logic as your training notebook)
import json
from torch.utils.data import Dataset

class HotpotQANeg(Dataset):
    def __init__(self, data, tokenizer, args, train: bool):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_len = args["max_len"]
        self.max_q_len = args["max_q_len"]
        self.max_q_sp_len = args["max_q_sp_len"]
        self.data = data
        self.train = train

    def encode_chunk_pair(self, t1, t2, max_len):
        return self.tokenizer(text=t1, text_pair=t2, max_length=max_len,
                              return_tensors="pt", padding=True, truncation=True)

    def encode_chunk(self, t, max_len):
        return self.tokenizer(text=t, max_length=max_len,
                              return_tensors="pt", padding=True, truncation=True)

    def __getitem__(self, index):
        d = self.data[index]
        question = d["question"]
        if question.endswith("?"):
            question = question[:-1]

        if d["type"] == "comparison":
            random.shuffle(d["pos_paras"])
            start_para, bridge_para = d["pos_paras"][0], d["pos_paras"][1]
        else:
            for para in d["pos_paras"]:
                if para["title"] != d["bridge"]:
                    start_para = para
                else:
                    bridge_para = para

        if self.train:
            random.shuffle(d["neg_paras"])

        c1_enc = self.encode_chunk_pair(start_para["title"].strip(), start_para["text"].strip(), self.max_len)
        c2_enc = self.encode_chunk_pair(bridge_para["title"].strip(), bridge_para["text"].strip(), self.max_len)

        n1_enc = self.encode_chunk_pair(d["neg_paras"][0]["title"].strip(), d["neg_paras"][0]["text"].strip(), self.max_len)
        n2_enc = self.encode_chunk_pair(d["neg_paras"][1]["title"].strip(), d["neg_paras"][1]["text"].strip(), self.max_len)

        q_enc = self.encode_chunk(question, max_len=self.max_q_len)
        q_c1_enc = self.encode_chunk_pair(question, start_para["text"].strip(), self.max_q_sp_len)

        return {"q_enc": q_enc, "q_c1_enc": q_c1_enc,
                "c1_enc": c1_enc, "c2_enc": c2_enc,
                "n1_enc": n1_enc, "n2_enc": n2_enc}

    def __len__(self):
        return len(self.data)

def collate_tokens(values, pad_idx, eos_idx=None, left_pad=False, move_eos_to_beginning=False):
    if len(values[0].size()) > 1:
        values = [v.view(-1) for v in values]
    size = max(v.size(0) for v in values)
    res = values[0].new(len(values), size).fill_(pad_idx)

    def copy_tensor(src, dst):
        assert dst.numel() == src.numel()
        if move_eos_to_beginning:
            assert src[-1] == eos_idx
            dst[0] = eos_idx
            dst[1:] = src[:-1]
        else:
            dst.copy_(src)

    for i, v in enumerate(values):
        copy_tensor(v, res[i][size - len(v):] if left_pad else res[i][:len(v)])
    return res

def Dataset_collate(samples):
    if len(samples) == 0:
        return {}
    batch = {
        "q_enc_btz": collate_tokens([s["q_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_mask": collate_tokens([s["q_enc"]["attention_mask"][0] for s in samples], 0),

        "q_c1_enc_btz": collate_tokens([s["q_c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_c1_mask": collate_tokens([s["q_c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c1_enc_btz": collate_tokens([s["c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c1_mask": collate_tokens([s["c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c2_enc_btz": collate_tokens([s["c2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c2_mask": collate_tokens([s["c2_enc"]["attention_mask"][0] for s in samples], 0),

        "n1_enc_btz": collate_tokens([s["n1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n1_mask": collate_tokens([s["n1_enc"]["attention_mask"][0] for s in samples], 0),

        "n2_enc_btz": collate_tokens([s["n2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n2_mask": collate_tokens([s["n2_enc"]["attention_mask"][0] for s in samples], 0),
    }
    return batch

def load_val_jsonl(path):
    # Filters out samples that cannot be evaluated (need >= 2 negatives)
    data = []
    with open(path, "r") as f:
        for line in f:
            d = json.loads(line)
            if "neg_paras" in d and len(d["neg_paras"]) >= 2:
                data.append(d)
    return data

In [ ]:
# Cell 6 (REPLACE): Retriever + evaluation (matches original train.py eval logic)
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModel
from tqdm import tqdm

class Retriever(nn.Module):
    def __init__(self, config, args_local):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(args_local["model_name"])
        self.args = args_local

        # Freeze layers if needed (same logic)
        self.freeze_encoder()

        self.project = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps),
        )

    def freeze_encoder(self):
        if self.args.get("freeze_layers", 0) > 0:
            freeze_layers = int(self.args["freeze_layers"])
            model_config = self.encoder.config
            if freeze_layers >= model_config.num_hidden_layers:
                for p in self.encoder.parameters():
                    p.requires_grad = False
            else:
                for name, p in self.encoder.named_parameters():
                    if name.startswith("encoder.layer"):
                        layer_index = int(name.split(".")[2])
                        if layer_index < freeze_layers:
                            p.requires_grad = False

    def encode_seq(self, input_ids, mask):
        cls_rep = self.encoder(input_ids, mask)[0][:, 0, :]  # CLS
        return self.project(cls_rep)

    def forward(self, batch):
        q_emb = self.encode_seq(batch["q_enc_btz"], batch["q_mask"])
        q_c1_emb = self.encode_seq(batch["q_c1_enc_btz"], batch["q_c1_mask"])

        c1_emb = self.encode_seq(batch["c1_enc_btz"], batch["c1_mask"])
        c2_emb = self.encode_seq(batch["c2_enc_btz"], batch["c2_mask"])

        n1_emb = self.encode_seq(batch["n1_enc_btz"], batch["n1_mask"])
        n2_emb = self.encode_seq(batch["n2_enc_btz"], batch["n2_mask"])

        return {
            "q_emb": q_emb, "q_c1_emb": q_c1_emb,
            "c1_emb": c1_emb, "c2_emb": c2_emb,
            "n1_emb": n1_emb, "n2_emb": n2_emb
        }

@torch.no_grad()
def mhop_eval_exact(embs):
    # Same logic as KGP/MDR/train.py:mhop_eval
    c_embs = torch.cat([embs["c1_emb"], embs["c2_emb"]], dim=0)  # (2B) x D
    n_embs = torch.cat([embs["n1_emb"].unsqueeze(1), embs["n2_emb"].unsqueeze(1)], dim=1)  # B x 2 x D

    scores_1 = torch.mm(embs["q_emb"], c_embs.t())  # B x 2B
    n_scores_1 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2

    scores_2 = torch.mm(embs["q_c1_emb"], c_embs.t())  # B x 2B
    # NOTE: repo uses q_emb here (not q_c1_emb) for n_scores_2
    n_scores_2 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2

    bsize = embs["q_emb"].size(0)
    scores_1_mask = torch.cat([torch.zeros(bsize, bsize), torch.eye(bsize)], dim=1).to(embs["q_emb"].device)
    scores_1 = scores_1.float().masked_fill(scores_1_mask.bool(), float("-inf")).type_as(scores_1)

    scores_1 = torch.cat([scores_1, n_scores_1], dim=1)  # B x (2B+2)
    scores_2 = torch.cat([scores_2, n_scores_2], dim=1)  # B x (2B+2)

    target_1 = torch.arange(bsize).to(embs["q_emb"].device)
    target_2 = torch.arange(bsize).to(embs["q_emb"].device) + bsize

    ranked_1_hop = scores_1.argsort(dim=1, descending=True)
    ranked_2_hop = scores_2.argsort(dim=1, descending=True)
    idx2ranked_1 = ranked_1_hop.argsort(dim=1)
    idx2ranked_2 = ranked_2_hop.argsort(dim=1)

    rrs_1, rrs_2 = [], []
    for t, idx2ranked in zip(target_1, idx2ranked_1):
        rrs_1.append(1.0 / (idx2ranked[t].item() + 1))
    for t, idx2ranked in zip(target_2, idx2ranked_2):
        rrs_2.append(1.0 / (idx2ranked[t].item() + 1))

    return rrs_1, rrs_2

@torch.no_grad()
def eval_mrr_exact(model, dataloader, device):
    model.eval()
    rrs_1_all, rrs_2_all = [], []
    for batch in tqdm(dataloader, desc="Eval"):
        batch = move_to_gpu(batch, device=device)
        embs = model(batch)
        r1, r2 = mhop_eval_exact(embs)
        rrs_1_all += r1
        rrs_2_all += r2
    return float(np.mean(rrs_1_all)), float(np.mean(rrs_2_all))

In [ ]:
# Cell 7: Locate val_with_neg_v0.json for HotpotQA and 2WikiMQA (auto-discovery)
import os

def find_val_files(base_dir):
    found = []
    for root, _, files in os.walk(os.path.join(base_dir, "DATA")):
        for fn in files:
            if fn == "val_with_neg_v0.json":
                found.append(os.path.join(root, fn))
    return sorted(found)

val_files = find_val_files(BASE_DIR)

print("Found val_with_neg_v0.json files:")
for p in val_files:
    print(" -", p)

# Map to dataset name by folder
dataset_to_val = {}
for p in val_files:
    norm = p.replace("\\", "/")
    if "/HotpotQA/" in norm:
        dataset_to_val["HotpotQA"] = p
    elif "/2WikiMQA/" in norm:
        dataset_to_val["2WikiMQA"] = p

print("\nDataset map:", dataset_to_val)

Found val_with_neg_v0.json files:
 - /content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json

Dataset map: {'HotpotQA': '/content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json'}


In [ ]:
# Cell 8 (FINAL FIX): Load checkpoints with key-renaming for your Retriever wrapper
import os
import numpy as np
import torch

torch.set_float32_matmul_precision("high")

def safe_torch_load(path, map_location="cpu", trust_checkpoint=True):
    # Fix PyTorch 2.6+ weights_only default
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass

    if trust_checkpoint:
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)
    else:
        try:
            return torch.load(path, map_location=map_location, weights_only=True)
        except TypeError:
            return torch.load(path, map_location=map_location)

def normalize_state_dict_keys(sd: dict) -> dict:
    """
    Make checkpoint keys compatible with your Retriever:
    - Strip 'module.' prefix
    - Strip leading '.' prefix
    - If keys look like HuggingFace backbone weights without 'encoder.' prefix,
      add 'encoder.' so they match Retriever.encoder.*
    """
    out = {}
    for k, v in sd.items():
        # 1) strip DataParallel prefix
        if k.startswith("module."):
            k = k[len("module."):]
        # 2) strip accidental leading dot
        if k.startswith("."):
            k = k[1:]

        # 3) If already correct, keep
        if k.startswith("encoder.") or k.startswith("project."):
            out[k] = v
            continue

        # 4) If checkpoint stores backbone weights directly (embeddings./encoder./pooler.)
        #    your wrapper expects them under encoder.*
        if k.startswith("embeddings.") or k.startswith("encoder.") or k.startswith("pooler."):
            out["encoder." + k] = v
            continue

        # 5) Otherwise keep as-is (safe fallback)
        out[k] = v

    return out

def extract_state_dict_clean(ckpt_obj):
    # training checkpoint dict vs raw state_dict
    if isinstance(ckpt_obj, dict) and "model_state_dict" in ckpt_obj:
        sd = ckpt_obj["model_state_dict"]
    else:
        sd = ckpt_obj

    if not isinstance(sd, dict):
        raise TypeError("Checkpoint is not a state_dict or does not contain model_state_dict.")

    sd = normalize_state_dict_keys(sd)
    return sd

def build_retriever(model_name, base_args):
    args_local = dict(base_args)
    args_local["model_name"] = model_name
    tok, cfg = load_tokenizer(model_name)
    model = Retriever(cfg, args_local).to(device)
    model.eval()
    return model, tok

def load_checkpoint_into(model, state_dict, strict=True):
    missing, unexpected = model.load_state_dict(state_dict, strict=strict)
    return list(missing), list(unexpected)

def acceptable_load(missing_keys, unexpected_keys):
    # Allow only very small mismatch; pooler-only missing is acceptable
    if len(missing_keys) == 0 and len(unexpected_keys) == 0:
        return True
    if len(unexpected_keys) <= 1 and len(missing_keys) <= 2:
        if all(("pooler" in k) for k in missing_keys):
            return True
    return False

def load_model_best_effort(ckpt_path, candidates, base_args, trust_checkpoint=True, require_acceptable=True):
    ckpt = safe_torch_load(ckpt_path, map_location="cpu", trust_checkpoint=trust_checkpoint)
    sd = extract_state_dict_clean(ckpt)

    best = None
    last_err = None

    for mname in candidates:
        try:
            model, tok = build_retriever(mname, base_args)

            # Prefer strict=True
            try:
                miss, unexp = load_checkpoint_into(model, sd, strict=True)
                info = {"model_name": mname, "strict": True, "missing": miss, "unexpected": unexp}
                return model, tok, info
            except Exception:
                miss, unexp = load_checkpoint_into(model, sd, strict=False)
                info = {"model_name": mname, "strict": False, "missing": miss, "unexpected": unexp}

                if require_acceptable and not acceptable_load(miss, unexp):
                    continue

                score = len(miss) * 10 + len(unexp)
                if (best is None) or (score < best[0]):
                    best = (score, model, tok, info)

        except Exception as e:
            last_err = e
            continue

    if best is not None:
        return best[1], best[2], best[3]

    raise RuntimeError(f"Could not load checkpoint properly. Last error: {last_err}")

# Paths
new_ckpt = os.path.join(BASE_DIR, "mdr_best_model.pt")
old_ckpt = os.path.join(BASE_DIR, "mdr_best_model_old.pt")
assert os.path.isfile(new_ckpt), f"Missing: {new_ckpt}"
assert os.path.isfile(old_ckpt), f"Missing: {old_ckpt}"

models = {}

# --- NEW: must match training backbone exactly ---
print("Loading NEW with config backbone:", args["model_name"])
new_model, new_tok = build_retriever(args["model_name"], args)

sd_new = extract_state_dict_clean(safe_torch_load(new_ckpt, map_location="cpu", trust_checkpoint=True))

try:
    miss, unexp = load_checkpoint_into(new_model, sd_new, strict=True)
    new_info = {"model_name": args["model_name"], "strict": True, "missing": miss, "unexpected": unexp}
except Exception as e:
    miss, unexp = load_checkpoint_into(new_model, sd_new, strict=False)
    if not acceptable_load(miss, unexp):
        raise RuntimeError(
            "NEW checkpoint did NOT load cleanly even after key-normalization.\n"
            f"missing={len(miss)}, unexpected={len(unexp)}\n"
            f"missing examples={miss[:5]}\nunexpected examples={unexp[:5]}\n"
            f"strict error={str(e)[:200]}"
        )
    new_info = {"model_name": args["model_name"], "strict": False, "missing": miss, "unexpected": unexp}

models["NEW"] = {"model": new_model, "tokenizer": new_tok, "info": new_info, "ckpt": new_ckpt}
print("NEW info:", {k: (len(v) if isinstance(v, list) else v) for k, v in new_info.items()})

# --- OLD: try candidates ---
OLD_CANDIDATES = ["bert-base-uncased", "roberta-base", args["model_name"]]
print("\nLoading OLD with candidates:", OLD_CANDIDATES)
old_model, old_tok, old_info = load_model_best_effort(old_ckpt, OLD_CANDIDATES, args, trust_checkpoint=True, require_acceptable=True)

models["OLD"] = {"model": old_model, "tokenizer": old_tok, "info": old_info, "ckpt": old_ckpt}
print("OLD info:", {k: (len(v) if isinstance(v, list) else v) for k, v in old_info.items()})

print("\nModels loaded successfully with normalized keys.")

Loading NEW with config backbone: deepset/tinyroberta-squad2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at deepset/tinyroberta-squad2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OLD info: {'model_name': 'bert-base-uncased', 'strict': False, 'missing': 0, 'unexpected': 1}

Models loaded successfully with normalized keys.


In [ ]:
# Cell 9 (REPLACE): Sanity check - ensure models dict is from Cell 8 (NEW)
def _assert_valid_models(models):
    assert isinstance(models, dict) and len(models) > 0, "models is empty. Run Cell 8 (NEW) first."
    for tag, pack in models.items():
        assert "model" in pack and "tokenizer" in pack and "info" in pack, f"{tag}: invalid pack structure"
        info = pack["info"]
        # We require the new (correct) schema
        assert "strict" in info and "missing" in info and "unexpected" in info and "model_name" in info, (
            f"{tag}: models['{tag}']['info'] is from OLD loader. "
            "Delete/skip the old Cell 9 and re-run Cell 8 (NEW)."
        )

_assert_valid_models(models)

print("Sanity check OK. Model infos:")
for tag, pack in models.items():
    info = pack["info"]
    print(f"- {tag}: backbone={info['model_name']} strict={info['strict']} missing={len(info['missing'])} unexpected={len(info['unexpected'])}")

Sanity check OK. Model infos:
- NEW: backbone=deepset/tinyroberta-squad2 strict=True missing=0 unexpected=0
- OLD: backbone=bert-base-uncased strict=False missing=0 unexpected=1


In [ ]:
# Cell 10 (REPLACE): Evaluate with exact MRR logic (repo-faithful)
from torch.utils.data import DataLoader
import pandas as pd
import torch

def evaluate_one(model, tokenizer, val_path, base_args):
    data = load_val_jsonl(val_path)
    ds = HotpotQANeg(data, tokenizer, base_args, train=False)

    for bsz in [int(base_args.get("eval_bsz", 32)), 16, 8, 4, 2, 1]:
        try:
            dl = DataLoader(
                ds,
                batch_size=bsz,
                pin_memory=True,
                collate_fn=Dataset_collate,
                num_workers=int(base_args.get("num_workers", 0)),
                shuffle=False,
            )
            mrr1, mrr2 = eval_mrr_exact(model, dl, device=device)
            return {
                "val_path": val_path,
                "num_samples": len(ds),
                "eval_bsz_used": bsz,
                "MRR_1": mrr1,
                "MRR_2": mrr2,
                "MRR_avg": (mrr1 + mrr2) / 2.0,
            }
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                continue
            raise

    raise RuntimeError("All batch sizes failed (OOM).")

rows = []
for ds_name, val_path in dataset_to_val.items():
    print(f"\n=== Dataset: {ds_name} ===")
    for tag, pack in models.items():
        print(f"\n-- Evaluating {tag} --")
        out = evaluate_one(pack["model"], pack["tokenizer"], val_path, args)

        info = pack["info"]
        row = {
            "dataset": ds_name,
            "model_tag": tag,
            "checkpoint": os.path.basename(pack["ckpt"]),
            "backbone_model_name": info["model_name"],
            "strict_load": info["strict"],
            "missing_keys": len(info["missing"]),
            "unexpected_keys": len(info["unexpected"]),
            **out,
        }
        rows.append(row)
        print(row)

df = pd.DataFrame(rows)
df


=== Dataset: HotpotQA ===

-- Evaluating NEW --


Eval: 100%|██████████| 232/232 [01:23<00:00,  2.79it/s]


{'dataset': 'HotpotQA', 'model_tag': 'NEW', 'checkpoint': 'mdr_best_model.pt', 'backbone_model_name': 'deepset/tinyroberta-squad2', 'strict_load': True, 'missing_keys': 0, 'unexpected_keys': 0, 'val_path': '/content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json', 'num_samples': 7405, 'eval_bsz_used': 32, 'MRR_1': 0.9383152805852228, 'MRR_2': 0.9048913594808657, 'MRR_avg': 0.9216033200330442}

-- Evaluating OLD --


Eval: 100%|██████████| 232/232 [02:16<00:00,  1.69it/s]

{'dataset': 'HotpotQA', 'model_tag': 'OLD', 'checkpoint': 'mdr_best_model_old.pt', 'backbone_model_name': 'bert-base-uncased', 'strict_load': False, 'missing_keys': 0, 'unexpected_keys': 1, 'val_path': '/content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json', 'num_samples': 7405, 'eval_bsz_used': 32, 'MRR_1': 0.9518809129721412, 'MRR_2': 0.9259723126299764, 'MRR_avg': 0.9389266128010587}


,dataset,model_tag,checkpoint,backbone_model_name,strict_load,missing_keys,unexpected_keys,val_path,num_samples,eval_bsz_used,MRR_1,MRR_2,MRR_avg
0,HotpotQA,NEW,mdr_best_model.pt,deepset/tinyroberta-squad2,True,0,0,/content/drive/MyDrive/final_project/baseline/...,7405,32,0.938315,0.904891,0.921603
1,HotpotQA,OLD,mdr_best_model_old.pt,bert-base-uncased,False,0,1,/content/drive/MyDrive/final_project/baseline/...,7405,32,0.951881,0.925972,0.938927
